# 計算機数学 II 中間レポート

[student number removed]
理工学群数学類3年　カウエ　シンゴ　アラマキ

Python 3.12.3

Windows 11

In [218]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install sympy
!{sys.executable} -m pip install scipy

import sympy as sp
import numpy as np

# Python 3.12.3
# Windows 11

問題1

In [214]:
#変数を定義する
x, y = sp.symbols('x y')

#関数を定義する
f1 = -x**3 + x + y**2 - 1
f2 = x**4 + x*y + y**4 - 1

#行列を立てる
F = sp.Matrix([f1, f2])
vars = sp.Matrix([x, y])

#ヤコビ行列
J = F.jacobian(vars)

#数値関数に変換
F_num = sp.lambdify((x, y), F, 'numpy')
J_num = sp.lambdify((x, y), J, 'numpy')

#ニュートン法の実装
def newton(x0, f, Df, tol=1e-15, max_iter=100):
    x1 = np.array(x0, dtype=float)

    for i in range(max_iter): # iter 100にbreak
        Fx = np.array(f(x1[0], x1[1]), dtype=float).flatten()
        Jx = np.array(Df(x1[0], x1[1]), dtype=float)

        # Jx * step = Fx を解く
        step = np.linalg.solve(Jx, Fx)
        x2 = x1 - step

        print(f"iter反複 {i}: {x2}")

        if np.linalg.norm(x2 - x1) < tol: #収束判定tolにbreak
            print(f"{i}回iter反複に収束")
            break

        x1 = x2

    return x2

newton([0, 1.1], F_num, J_num)


iter反複 0: [-0.03340909  1.0197314 ]
iter反複 1: [-7.57323154e-04  1.00059756e+00]
iter反複 2: [-9.73631173e-07  1.00000067e+00]
iter反複 3: [-1.1231898e-12  1.0000000e+00]
iter反複 4: [-8.81856621e-17  1.00000000e+00]
iter反複 5: [2.28366404e-17 1.00000000e+00]
5回iter反複に収束


array([2.28366404e-17, 1.00000000e+00])

step6にてbreakされたので、[0,1.1] が初期条件では5反複目でtolに満たせたので反複列は収束する

問題2

In [215]:
# funcを定義する

# Aとbを返す関数
def return_A(n):
    A = 2*np.eye(n) - np.diag(np.ones(n-1), k=1) - np.diag(np.ones(n-1), k=-1)
    return A
def return_b(n):
    b = np.zeros(n)
    b[0] = 1
    return b

# CG法の実装
def cg(A, b, x0=None, tol=1e-15, max_iter=150, verbose=False):
    # 初期化
    if x0 is None:
        x0 = np.zeros_like(b)  # now it will always match the size of b
    x = x0.copy()
    r = b - A @ x
    p = r.copy()
    rsold = r @ r

    # 反複
    for k in range(max_iter): # max_iterにbreak
        Ap = A @ p
        alpha = rsold / (p @ Ap)
        x = x + alpha * p
        r = r - alpha * Ap
        rsnew = r @ r
        
        if verbose:
            print(f"iter反複 {k+1}: {x}")

        # 収束判定にbreak
        if np.sqrt(rsnew) < tol * np.linalg.norm(b):
            print(f"tol満たせた{k+1}回iter反複に収束")
            break
        
        beta = rsnew / rsold
        p = r + beta * p
        rsold = rsnew
        
    return x

In [216]:
import numpy as np
# n=5の場合
n = 5
A = return_A(n)
b = return_b(n)

x = cg(A, b, verbose=True)
print(x)
print(x*6)


iter反複 1: [0.5 0.  0.  0.  0. ]
iter反複 2: [0.66666667 0.33333333 0.         0.         0.        ]
iter反複 3: [0.75 0.5  0.25 0.   0.  ]
iter反複 4: [0.8 0.6 0.4 0.2 0. ]
iter反複 5: [0.83333333 0.66666667 0.5        0.33333333 0.16666667]
tol満たせた5回iter反複に収束
[0.83333333 0.66666667 0.5        0.33333333 0.16666667]
[5. 4. 3. 2. 1.]


5回目の反複にて解xが tolに満たし、1/6 * [5 4 3 2 1]に収束した。

In [217]:
for n in [50, 52, 67, 69, 78, 100, 101]:
    An = return_A(n)
    bn = return_b(n)

    x = cg(An, bn, max_iter=5*n, tol=1e-10)
    print(f"n={n}のとき、x*(n+1): {x*(n+1)}")
    print(f"\n")

tol満たせた50回iter反複に収束
n=50のとき、x*(n+1): [50. 49. 48. 47. 46. 45. 44. 43. 42. 41. 40. 39. 38. 37. 36. 35. 34. 33.
 32. 31. 30. 29. 28. 27. 26. 25. 24. 23. 22. 21. 20. 19. 18. 17. 16. 15.
 14. 13. 12. 11. 10.  9.  8.  7.  6.  5.  4.  3.  2.  1.]


tol満たせた52回iter反複に収束
n=52のとき、x*(n+1): [52. 51. 50. 49. 48. 47. 46. 45. 44. 43. 42. 41. 40. 39. 38. 37. 36. 35.
 34. 33. 32. 31. 30. 29. 28. 27. 26. 25. 24. 23. 22. 21. 20. 19. 18. 17.
 16. 15. 14. 13. 12. 11. 10.  9.  8.  7.  6.  5.  4.  3.  2.  1.]


tol満たせた67回iter反複に収束
n=67のとき、x*(n+1): [67. 66. 65. 64. 63. 62. 61. 60. 59. 58. 57. 56. 55. 54. 53. 52. 51. 50.
 49. 48. 47. 46. 45. 44. 43. 42. 41. 40. 39. 38. 37. 36. 35. 34. 33. 32.
 31. 30. 29. 28. 27. 26. 25. 24. 23. 22. 21. 20. 19. 18. 17. 16. 15. 14.
 13. 12. 11. 10.  9.  8.  7.  6.  5.  4.  3.  2.  1.]


tol満たせた69回iter反複に収束
n=69のとき、x*(n+1): [69. 68. 67. 66. 65. 64. 63. 62. 61. 60. 59. 58. 57. 56. 55. 54. 53. 52.
 51. 50. 49. 48. 47. 46. 45. 44. 43. 42. 41. 40. 39. 38. 37. 36. 35. 34.
 33. 32. 31

考察として、A_n * x = b_n の一般な解 x は x = [n n-1 ... 2 1] * (1/n+1) となる。